# HW2 前處理：補齊 HW1 index 的缺口

HW1 的 `hw1_indexer.py` 產生的 `dictionary.txt` / `postings.txt`（由 `build.py index` 從 WARC 建出）少了 HW2 排序需要的東西。這個 notebook 掃 `postings.txt` 兩次、WARC 兩次把它們補出來，**不需要重建 index**。

| 缺口 | 為什麼 HW2 需要 | 輸出檔 |
|---|---|---|
| 每個 term 在 `postings.txt` 的 byte offset | HW1 的 trie offset 只存在 build 當下的記憶體裡，HW2 要能直接 `seek()` 讀單一 term 的 postings | `output/offsets.tsv` |
| 文件總數 N | idf = log(N / df) | `output/meta.json` |
| 每份文件的 vector 長度 ‖d‖ | cosine 的分母，query 時算不出來 | `output/doc_norms.tsv` |
| 每份文件的長度 \|d\| 與平均長度 avgdl | BM25 的長度正規化 | `output/doc_lengths.tsv`、`meta.json` |
| champion lists | inexact top-k：只對少數文件計分 | `output/champions.tsv`、`output/champions_bm25.tsv` |
| 原始字形（surface form）→ term 的對照 | wildcard 比對原始字形（`*ing`）、spelling correction 的字典 | `output/surface_vocab.tsv` |
| docID → TREC-ID 對照（選用） | 助教若用 ClueWeb09 qrels 對答案，需要原始 ID | `output/doc_ids.tsv` |

權重公式依投影片：$w_{t,d} = (1 + \log_{10} tf_{t,d}) \cdot \log_{10}(N / df_t)$

**和 `prep.py` 的關係**：實際的程式都寫在 `prep.py` 的函式裡，這個 notebook 一步一步呼叫它們，並把每一步的推導寫出來。直接執行 `python prep.py` 會依同樣的順序呼叫同樣的函式，所以兩種方式產生的檔案完全相同，只需要跑其中一種。

**放置方式**：本 notebook 和 `prep.py`、`hw1_indexer.py`、`output/` 放在同一個資料夾。若同資料夾有 `07.warc.gz`，N 和 TREC-ID 會從 WARC 精確取得；沒有的話 N 退而用 postings 中最大的 docID，並在 `meta.json` 標註來源。

## 1. 設定

所有路徑與參數集中在這裡，預設值和 `prep.py` 相同。`LOG_BASE` 會寫進 `meta.json`，HW2 查詢端必須用同一個底數，否則 query 和 document 的權重尺度會不一致。

In [1]:
from pathlib import Path

# folder holding prep.py, hw1_indexer.py, output/ and (optionally) the WARC file
BASE_DIR = Path(".").resolve()
# folder with HW1's dictionary.txt and postings.txt; new files are written here too
INDEX_DIR = BASE_DIR / "output"
# HW1 postings file: one record per line, "term, df: <doc, tf: <pos, ...>; ...>"
POSTINGS_PATH = INDEX_DIR / "postings.txt"
# HW1 dictionary file: one "term, df" per line, same term order as postings.txt
DICTIONARY_PATH = INDEX_DIR / "dictionary.txt"
# optional ClueWeb09 WARC file; when it exists, N and TREC-IDs are taken from it
WARC_PATH = BASE_DIR / "07.warc.gz"
# logarithm base used by both tf weight and idf (Slide 4 formula)
LOG_BASE = 10
# champion list orderings to build ("cosine" -> champions.tsv, "bm25" -> champions_bm25.tsv)
CHAMPION_SCHEMES = ["cosine", "bm25"]
# documents kept per term in a champion list
CHAMPION_R = 50
# BM25 k1 used to order champions_bm25.tsv
BM25_K1 = 1.2
# BM25 b used to order champions_bm25.tsv
BM25_B = 0.75
# rebuild surface_vocab.tsv from the WARC (about 4 minutes); False keeps an existing file
BUILD_SURFACE = True
# number of random terms whose seek is checked in 10.2
NUM_SAMPLE_TERMS = 2000
# number of random documents whose norm and length are recomputed in 10.3
NUM_SAMPLE_DOCS = 20
# number of random terms whose champion lists are rebuilt in 10.4
NUM_SAMPLE_CHAMPION_TERMS = 300
# fixed seed so the verification samples are reproducible
RANDOM_SEED = 27

## 2. 匯入

`prep.py` 會 import `hw1_indexer`，直接重用 HW1 的程式，而不是複製一份：

- `iter_warc_records`：HW1 已經處理過截斷 gzip、錯誤 Content-Length 等狀況的 WARC reader。
- `parse_posting_line`：驗證區用它解析 seek 讀到的 record，確認讀出的內容和 HW1 的格式定義一致。
- `Config`：記錄 build index 時的 linguistic 設定，HW2 查詢端要用同一份設定處理 query。

In [2]:
import sys
import json
import random
import time

# make prep.py and hw1_indexer.py in BASE_DIR importable
sys.path.insert(0, str(BASE_DIR))
# every preprocessing step lives in prep.py
import prep

# fail early with a clear message if postings.txt is not where expected
assert POSTINGS_PATH.exists(), f"postings not found: {POSTINGS_PATH}"
# fail early with a clear message if dictionary.txt is not where expected
assert DICTIONARY_PATH.exists(), f"dictionary not found: {DICTIONARY_PATH}"
# report whether the WARC file is available (decides how N is obtained)
print("WARC available:", WARC_PATH.exists())

WARC available: True


## 3. 從 WARC 取得 N 與 docID → TREC-ID（選用）

HW1 的 `parse_warc` 對**每一筆** `WARC-Type: response` record 依序編號（從 1 開始），即使抽不出任何文字也會佔一個 docID。所以：

- **N = response record 的數量**，這和 HW1 build 時的 `num_docs` 相同。
- 用 postings 中最大的 docID 當 N 只是下界：檔案尾端若有抽不出 token 的文件，它們不會出現在任何 postings 裡。

`prep.scan_warc_doc_ids` 只讀 WARC header，不解析 HTML，所以比 HW1 build 快很多。編號規則必須和 `parse_warc` 完全一致，否則 TREC-ID 會錯位。

沒有 WARC 時這格會跳過，`warc_docs` 維持 `None`。

In [3]:
# None means "no WARC, fall back to max doc_id later"
warc_docs = None
# only scan when the WARC file is present
if WARC_PATH.exists():
    # start timer for the scan
    t0 = time.time()
    # number every response record like hw1.parse_warc
    warc_docs = prep.scan_warc_doc_ids(str(WARC_PATH))
    # report the record count and elapsed time
    print(f"response records: {len(warc_docs):,}  ({time.time() - t0:.1f}s)")
    # show the first entry, to eyeball that TREC-IDs look right
    print("first:", warc_docs[0])
    # show the last entry, which should match the end of the collection
    print("last :", warc_docs[-1])
# no WARC in BASE_DIR
else:
    # explain the fallback that section 5 will use
    print("WARC not found, skipping; N will fall back to the max doc_id in postings.txt")

response records: 31,480  (1.6s)
first: (1, 'clueweb09-en0000-07-00000', 'http://000strega000.iobloggo.com/')
last : (31480, 'clueweb09-en0000-07-31479', 'http://cid-33c5dc2990e3a4dd.profile.live.com/')


## 4. 第一次掃描 postings.txt：offset、norm 的累加和、文件長度

`prep.scan_postings` 逐行以 binary 模式讀取。HW1 以 binary 寫檔、每個 term 一行（`multiline=False`），所以「目前累計的 byte 數」就是該行開頭的 offset，在 Windows 上也不會因換行轉換而偏移。

每行只取出 `(docID, tf)`，不解析 positions，這樣比呼叫 `parse_posting_line` 快很多。用的 regex 是 `(\d+), (\d+): <`：positions 清單裡只有 `數字, 數字` 和 `>`，不會出現 `: <`，所以只會命中每個 posting 的開頭。每行都會檢查命中數等於 df，格式有任何意外都會立刻報錯。

**為什麼可以在還不知道 N 的情況下累加 document norm？** 令 $a = 1 + \log tf$、$L = \log N$、$\ell = \log df$，則

$$\|d\|^2 = \sum_t a_t^2 (L - \ell_t)^2 = L^2 \sum a^2 \;-\; 2L \sum a^2\ell \;+\; \sum a^2\ell^2$$

所以每份文件只要累加三個和 $S_0=\sum a^2$、$S_1=\sum a^2\ell$、$S_2=\sum a^2\ell^2$，等 N 確定後再一次算出 norm。

**文件長度 $|d|$**：同一次掃描順便累加每份文件的 $\sum_t tf_{t,d}$，也就是經過 stopword 移除、stemming 之後留在 index 裡的 token 數。BM25 用它做長度正規化。

In [4]:
# start timer for the scan
t0 = time.time()
# offsets, the three norm sums and |d| in one pass
scan = prep.scan_postings(str(POSTINGS_PATH), LOG_BASE)
# vocabulary size
print(f"terms           : {len(scan.term_entries):,}")
# total (term, doc) pairs
print(f"postings        : {scan.num_postings:,}")
# documents that have at least one indexed term
print(f"docs with terms : {len(scan.s0):,}")
# bytes read, equal to the size of postings.txt
print(f"file bytes      : {scan.postings_bytes:,}")
# scan duration
print(f"elapsed         : {time.time() - t0:.1f}s")

terms           : 346,789
postings        : 8,976,349
docs with terms : 31,480
file bytes      : 201,085,677
elapsed         : 11.1s


## 5. 決定 N，計算 document norm 與 avgdl

- 有 WARC：N = response record 數，並檢查 postings 中最大的 docID 沒有超過 N（超過代表 WARC 和 index 不是同一份資料）。
- 沒 WARC：N = 最大 docID，`N_source` 記為 `max_doc_id`，提醒這是下界。

浮點誤差可能讓展開式算出極小的負數（例如某份文件所有 term 的 df 都等於 N，idf 全為 0），所以先夾到 0 再開根號。norm 為 0 的文件在 cosine 時要跳過，避免除以 0。

avgdl 是 $\sum_d |d| / N$，沒有任何 term 的文件以長度 0 計入。

In [5]:
# N, where it came from, and the largest doc_id in the index
N, N_source, max_doc_id = prep.decide_n(scan, warc_docs)
# doc_id -> ||d|| from the expanded formula
doc_norms = prep.compute_doc_norms(scan, N, LOG_BASE)
# BM25 average document length
avgdl = prep.average_doc_length(scan.doc_lengths, N)

# documents whose every term has idf 0 (cosine must skip them)
zero_norm_docs = sum(1 for v in doc_norms.values() if v == 0.0)
# chosen N and its source
print(f"N = {N:,} (source: {N_source})")
# largest doc_id seen in the index
print(f"max doc_id in postings = {max_doc_id:,}")
# documents numbered by HW1 but absent from every postings list
print(f"docs without any term  = {N - len(doc_norms):,}")
# documents with zero vector length
print(f"docs with zero norm    = {zero_norm_docs:,}")
# average document length
print(f"avgdl                  = {avgdl:.2f}")

N = 31,480 (source: warc)
max doc_id in postings = 31,480
docs without any term  = 0
docs with zero norm    = 0
avgdl                  = 568.37


## 6. 寫出表格檔

| 檔案 | 格式 | 說明 |
|---|---|---|
| `offsets.tsv` | `term \t df \t byte_offset` | 順序和 `dictionary.txt` 相同 |
| `doc_norms.tsv` | `doc_id \t norm` | 只列出有 term 的文件；不在表中的文件 norm 視為 0 |
| `doc_lengths.tsv` | `doc_id \t length` | 只列出有 term 的文件；不在表中的文件長度視為 0 |
| `doc_ids.tsv` | `doc_id \t trec_id \t url` | 只有提供 WARC 時才會產生 |

用 tab 當分隔符號，因為 HW1 的 tokenizer 以空白切詞，term 裡不會有 tab；寫檔前仍會檢查。所有檔案都以 `newline="\n"` 寫出，在 Windows 和 Linux 上內容逐 byte 相同。

In [6]:
# offsets / norms / lengths / (with the WARC) doc_ids
prep.write_tables(str(INDEX_DIR), scan, doc_norms, warc_docs)
# list the files that were written
for name in ("offsets.tsv", "doc_norms.tsv", "doc_lengths.tsv", "doc_ids.tsv"):
    # doc_ids.tsv may not exist
    if (INDEX_DIR / name).exists():
        # name and size of the file
        print(f"{name:16s} {(INDEX_DIR / name).stat().st_size:>12,} bytes")

offsets.tsv         6,889,896 bytes
doc_norms.tsv         758,973 bytes
doc_lengths.tsv       305,585 bytes
doc_ids.tsv         2,792,553 bytes


## 7. 第二次掃描 postings.txt：champion lists

課本 Ch. 7 的 champion list：每個 term 預先挑出「這個 term 貢獻最大」的前 r 份文件。查詢時只對所有 query term 的 champion list 聯集計分，候選文件從上萬份降到一兩百份。

**排序依據只需要 term 內部的比較**：同一個 term 的所有 posting 共用同一個 idf，所以 idf 不影響排序，可以省略。

| scheme | 輸出檔 | 每份文件的排序分數 | 需要的統計量 |
|---|---|---|---|
| `cosine` | `champions.tsv` | $(1+\log tf)\,/\,\|d\|$ | 第 5 步的 norm |
| `bm25` | `champions_bm25.tsv` | $\dfrac{tf\,(k_1+1)}{tf + k_1\,(1-b+b\,\lvert d\rvert/\text{avgdl})}$ | 第 4、5 步的 $\lvert d\rvert$ 和 avgdl |

細節：

- 分數相同時 docID 小的優先，所以每次產生的檔案都一樣。
- df ≤ r 的 term 直接保留全部文件。**每個 term 都會有一行**，ranker 查不到 term 時才不會誤以為它沒有任何文件。
- 每行格式是 `term \t doc doc ...`，doc 以遞增順序排列，行的順序和 `dictionary.txt` 相同。
- `champions.tsv` 是 `ranker.py` 的 `--champions` 目前讀取的檔案。`champions_bm25.tsv` 留給 BM25 計分使用。
- BM25 版本的排序綁定這裡的 $k_1$、$b$，所以會記進 `meta.json`。查詢時若改用不同的參數，champion list 仍然可以用，只是挑出的文件和實際計分的標準不完全一致。
- 沒有選到的 scheme，舊的輸出檔會被刪除，避免留下和這次 `meta.json` 不一致的檔案。

In [7]:
# start timer for the second pass
t0 = time.time()
# stream one champion file per scheme
champion_counts = prep.write_champions(str(POSTINGS_PATH), str(INDEX_DIR), CHAMPION_SCHEMES, CHAMPION_R,
                                       doc_norms, scan.doc_lengths, N, avgdl, BM25_K1, BM25_B, LOG_BASE)
# ids written per scheme
for scheme in CHAMPION_SCHEMES:
    # file name of this scheme
    name = prep.CHAMPION_FILES[scheme]
    # id count and size
    print(f"{name:20s} {champion_counts[scheme]:>10,} ids  {(INDEX_DIR / name).stat().st_size:>12,} bytes")
# pass duration
print(f"elapsed: {time.time() - t0:.1f}s")

champions.tsv         1,827,355 ids    13,046,407 bytes
champions_bm25.tsv    1,827,355 ids    13,032,895 bytes
elapsed: 10.3s


## 8. 原始字形詞表（surface vocabulary）

index 裡存的是 stem 之後的 term，原本的字形已經不見了：`running`、`runs` 都變成 `run`，所以 `*ing` 在 index 上找不到任何東西，拼錯的字也沒有正確拼法可以比對。這一步重新解析 WARC 的每一頁，用 HW1 的 `tokenize` 和 `linguistic`，記下每個被 index 收進去的 token 原本的字形（只做過 case folding）和它變成的 term：

| 欄位 | 意義 |
|---|---|
| surface | case folding 後的原始 token，例如 `running` |
| term | 這個 token 在 index 裡對應的 term，例如 `run` |
| cf | 這個字形在整個 collection 出現幾次 |
| df | 幾份文件含有這個字形 |

- 檔案依 surface 的 code point 排序，同樣開頭的字形是連續的一段，查詢時可以用 bisect。
- stopword 不會進 index，所以也不在表裡。
- **只保留 term 存在於 index 的字形。** 在這個環境重新解析 WARC，結果和交出去的 HW1 index 不完全相同：抽查前 300 份文件，有 11 份不同，差異集中在混合文字的亂碼 token 和少數 HTML 殘留，所以會多出約兩千組 index 裡沒有的 term。這些 term 查不到任何 postings，留著沒有用。第 10.5 節會印出這份詞表涵蓋 index 的比例。
- 沒有 WARC 或 `BUILD_SURFACE = False` 時不會重建；若既有的檔案是針對同一份 `postings.txt` 產生的，會沿用它。

In [8]:
# start timer for the full re-parse
t0 = time.time()
# build surface_vocab.tsv, or keep a still-valid existing one
surface_info = prep.surface_step(str(INDEX_DIR), str(WARC_PATH) if warc_docs is not None else None, scan, BUILD_SURFACE)
# what was built or kept
print(surface_info)
# step duration
print(f"elapsed: {time.time() - t0:.1f}s")

{'file': 'surface_vocab.tsv', 'pairs': 402109, 'dropped_not_in_index': 0}
elapsed: 160.0s


## 9. 寫出 meta.json

`meta.json` 最後才寫，所以它描述的一定是旁邊這一批檔案。`postings_bytes` 讓 `index_reader` 啟動時可以比對，避免 index 重建後還拿舊的 offset 去 seek。

| 欄位 | 用途 |
|---|---|
| `N`、`N_source`、`log_base` | idf 和 tf weight |
| `config` | HW1 build 時的 `Config`，query 要用同一份設定處理 |
| `postings_bytes` | 偵測過期的 offsets |
| `avgdl`、`total_doc_length` | BM25 |
| `champions` | r，以及每個 scheme 的檔名（BM25 另有 $k_1$、$b$） |
| `surface_vocab` | 原始字形詞表的檔名和行數；`None` 代表沒有可用的詞表 |

In [9]:
# global statistics and settings
meta = prep.build_meta(scan, doc_norms, N, N_source, max_doc_id, avgdl, LOG_BASE, warc_docs is not None,
                       CHAMPION_SCHEMES, CHAMPION_R, BM25_K1, BM25_B, surface_info)
# write it next to the tables
prep.write_meta(str(INDEX_DIR), meta)
# show it
print(json.dumps(meta, ensure_ascii=False, indent=2))

{
  "N": 31480,
  "N_source": "warc",
  "log_base": 10,
  "num_terms": 346789,
  "num_postings": 8976349,
  "num_docs_with_terms": 31480,
  "max_doc_id": 31480,
  "postings_bytes": 201085677,
  "config": {
    "include_title": true,
    "hyphen_mode": "split",
    "pos_base": 0,
    "case_folding": true,
    "stopword_removal": true,
    "lemmatization": false,
    "pos_tagging": false,
    "stemming": true,
    "use_trie": true,
    "use_skip_pointers": true,
    "spimi_block_limit": 1000000
  },
  "has_doc_ids": true,
  "avgdl": 568.3745235069886,
  "total_doc_length": 17892430,
  "champions": {
    "r": 50,
    "schemes": {
      "cosine": {
        "file": "champions.tsv"
      },
      "bm25": {
        "file": "champions_bm25.tsv",
        "k1": 1.2,
        "b": 0.75
      }
    }
  },
  "surface_vocab": {
    "file": "surface_vocab.tsv",
    "pairs": 402109,
    "dropped_not_in_index": 0
  }
}


## 10. 驗證

這一節以 HW1 的檔案和直接套公式的暴力計算當作正確答案，檢查的是**這個 notebook 自己的輸出**。各項都通過，才代表輸出檔可以安心給 HW2 用。

### 10.1 offsets.tsv 和 dictionary.txt 一致

逐行比對 term 與 df。兩者都是 HW1 merge 時同步寫出的，順序必須完全相同；數量或任何一行不同都代表掃描有誤。

In [10]:
# reproducible random generator, shared by 10.2 - 10.4
rng = random.Random(RANDOM_SEED)
# line-by-line comparison
checked = prep.verify_dictionary(str(DICTIONARY_PATH), scan)
# summary
print(f"OK: {checked:,} terms match dictionary.txt (term and df)")

OK: 346,789 terms match dictionary.txt (term and df)


### 10.2 用 offset seek 讀回的 record 正確

隨機抽 `NUM_SAMPLE_TERMS` 個 term，另外固定加上第一個和最後一個 term（邊界最容易出錯）。對每個 term：`seek(offset)` → 讀一行 → 交給 HW1 的 `parse_posting_line` 解析，檢查 term、df、postings 數量都相符。這等於模擬 HW2 查詢時的實際讀取路徑。

In [11]:
# seek + HW1 parser on sampled terms
checked = prep.verify_seeks(str(POSTINGS_PATH), scan, NUM_SAMPLE_TERMS, rng)
# summary
print(f"OK: {checked:,} seeks returned the right record")

OK: 2,002 seeks returned the right record


### 10.3 document norm 與文件長度和暴力計算一致

隨機抽 `NUM_SAMPLE_DOCS` 份文件，再加上 docID 最小與最大的文件。重新掃一次 postings，這次用 HW1 的 `parse_posting_line` 解析（不經過第 4 步的 regex），**直接**用 $\sum_t \big((1+\log tf)\cdot\log(N/df)\big)^2$ 計算 norm，並直接加總 tf 得到長度。norm 的相對誤差要小於 $10^{-9}$，長度必須完全相同。這一項驗證的是展開式的推導和實作，而不只是檔案格式。

In [12]:
# direct formulas on sampled documents
worst = prep.verify_doc_stats(str(POSTINGS_PATH), doc_norms, scan.doc_lengths, N, NUM_SAMPLE_DOCS, rng, LOG_BASE)
# summary
print(f"OK: lengths identical, worst relative norm error {worst:.2e}")

OK: lengths identical, worst relative norm error 9.09e-15


### 10.4 champion list 和完整排序一致

隨機抽 `NUM_SAMPLE_CHAMPION_TERMS` 個 term，再加上 df 最大的 20 個（真正需要截斷的情況）。對每個 term 用 HW1 的 parser 讀出全部 postings，直接套上表的公式**完整排序**後取前 r 名，和檔案中的內容比對。同時檢查每個檔案的行數和行順序都和 `dictionary.txt` 相同。

In [13]:
# full sort with the textbook formulas on sampled terms
checked = prep.verify_champions(str(POSTINGS_PATH), str(INDEX_DIR), scan, CHAMPION_SCHEMES, CHAMPION_R,
                                doc_norms, scan.doc_lengths, N, avgdl, BM25_K1, BM25_B,
                                NUM_SAMPLE_CHAMPION_TERMS, rng, LOG_BASE)
# summary
print(f"OK: champion lists of {checked} sampled terms match a full sort ({', '.join(CHAMPION_SCHEMES)})")

OK: champion lists of 320 sampled terms match a full sort (cosine, bm25)


### 10.5 原始字形詞表

逐行檢查：surface 嚴格遞增（排序正確、沒有重複）、每個 term 都在 index 裡、HW1 的 `linguistic` 把這個 token 處理後剛好得到這個 term、計數合理（$0 < df \le cf$）。最後印出這份詞表涵蓋了多少 index term，以及多少 token 出現次數。

In [14]:
# only when a surface vocabulary exists
if surface_info is not None:
    # structure, mapping and coverage checks
    info = prep.verify_surface_vocab(str(INDEX_DIR), scan, prep.hw1.Config())
    # summary
    print(f"OK: {info['pairs']:,} surface forms; covers {info['index_terms_covered']:.2%} of index terms "
          f"and {info['cf_covered']:.2%} of token occurrences")
# nothing to check
else:
    # explain
    print("no surface vocabulary")

OK: 402,109 surface forms; covers 100.00% of index terms and 100.00% of token occurrences


## 11. 完成

`output/` 現在多了 `offsets.tsv`、`doc_norms.tsv`、`doc_lengths.tsv`、`champions.tsv`、`champions_bm25.tsv`、`meta.json`（以及有 WARC 時的 `doc_ids.tsv`、`surface_vocab.tsv`）。HW2 的 `index_reader` 只需要讀這些小檔案進記憶體，postings 本身維持在硬碟上，按 offset 隨讀隨用。

若 `meta.json` 的 `N_source` 是 `max_doc_id`，表示這次沒有讀到 WARC：把 `07.warc.gz` 放到同一個資料夾後重跑整個 notebook，N 和 norm 都會更新為精確值。